# 01 — Baseline de IOI

Este primeiro notebook estabelece o comportamento que os notebooks seguintes tentarão explicar: se o GPT-2 Small resolve **Indirect Object Identification (IOI)** nestes prompts.

Regra do exemplo: se um nome aparece novamente como sujeito da ação, o outro nome é a resposta esperada.

> Exemplo: “John and Mary ... John gave the bag to” → resposta correta: **Mary**.

A medida central será:

$$\text{logit difference}=\text{logit(nome correto)}-\text{logit(nome incorreto)}$$

Um valor positivo favorece o nome correto. Ao final, teremos o baseline e a indexação dos tokens de resposta que o Notebook 02 usará para decompor esse resultado.

In [1]:
# Reunimos os utilitários usados para construir o baseline e conferir seu cálculo manualmente.
import torch

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    select_device,
    tokenize_answers,
)

# `device` fixa onde modelo e tensores viverão; as células seguintes reutilizam essa escolha.
device = select_device()
print(f"Dispositivo selecionado: {device}")

Dispositivo selecionado: cuda


In [2]:
# A escolha de `device` da célula anterior é usada ao carregar os pesos do GPT-2 Small.
model = load_gpt2_small(device)

# `model_shape` registra camada, heads por camada e largura do residual stream, nessa ordem.
model_shape = (
    model.cfg.n_layers,
    model.cfg.n_heads,
    model.cfg.d_model,
)
# A checagem evita interpretar os eixos seguintes com uma arquitetura diferente da esperada.
assert model_shape == (12, 12, 768)
print("camadas × heads por camada × dimensões:", model_shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

camadas × heads por camada × dimensões: (12, 12, 768)


In [3]:
# Construímos os oito exemplos e preservamos, para cada prompt, a ordem [resposta correta, incorreta].
prompts, answers = build_ioi_examples()
# `tokens`: (batch, position); `answer_tokens`: (batch, opção), com opção 0=correta e 1=incorreta.
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)

# A inspeção liga cada linha do batch aos IDs que serão comparados nos logits finais.
for index, (prompt, (correct, incorrect)) in enumerate(zip(prompts, answers)):
    print(f"{index}: {prompt!r}")
    print(f"   correto={correct!r} | incorreto={incorrect!r}")

# Esses shapes preparam a leitura dos logits como (batch, position, vocabulary) na próxima etapa.
print("\ntokens:", tuple(tokens.shape))
print("answer_tokens:", tuple(answer_tokens.shape))

0: 'When John and Mary went to the shops, John gave the bag to'
   correto=' Mary' | incorreto=' John'
1: 'When John and Mary went to the shops, Mary gave the bag to'
   correto=' John' | incorreto=' Mary'
2: 'When Tom and James went to the park, James gave the ball to'
   correto=' Tom' | incorreto=' James'
3: 'When Tom and James went to the park, Tom gave the ball to'
   correto=' James' | incorreto=' Tom'
4: 'When Dan and Sid went to the shops, Sid gave an apple to'
   correto=' Dan' | incorreto=' Sid'
5: 'When Dan and Sid went to the shops, Dan gave an apple to'
   correto=' Sid' | incorreto=' Dan'
6: 'After Martin and Amy went to the park, Amy gave a drink to'
   correto=' Martin' | incorreto=' Amy'
7: 'After Martin and Amy went to the park, Martin gave a drink to'
   correto=' Amy' | incorreto=' Martin'

tokens: (8, 15)
answer_tokens: (8, 2)


## O que o modelo devolve

Os logits terão três eixos:

$$\text{logits.shape}=(\text{batch},\text{sequence},\text{vocabulary})$$

- **batch**: os 8 prompts;
- **sequence**: as posições dos tokens de cada prompt;
- **vocabulary**: uma pontuação para cada token possível.

Como queremos prever o token seguinte, usaremos apenas a última posição da sequência.

In [4]:
# Usamos os `tokens` recém-construídos; o cache fica disponível para análises posteriores, sem alterar a saída.
with torch.inference_mode():
    logits, cache = model.run_with_cache(tokens)

# `logits` é (batch, position, vocabulary); `-1` escolhe a posição que prevê o próximo token.
print("logits:", tuple(logits.shape))
print("última posição:", tuple(logits[:, -1, :].shape))
# A próxima célula reunirá, nesse último eixo, apenas os IDs correto e incorreto de cada batch.

logits: (8, 15, 50257)
última posição: (8, 50257)


In [5]:
# A função usa os logits da última posição e os dois IDs de resposta associados a cada linha do batch.
# `per_prompt` mantém shape (batch,); `average` reduz esses oito valores a um escalar.
per_prompt = logits_to_logit_diff(
    logits,
    answer_tokens,
    per_prompt=True,
)
average = logits_to_logit_diff(logits, answer_tokens)

# Ver os exemplos separadamente impede que a média esconda um prompt com sinal contrário.
for index, difference in enumerate(per_prompt.tolist()):
    favored = "correto" if difference > 0 else "incorreto"
    print(f"prompt {index}: {difference:+.3f} → favorece o nome {favored}")

# A célula seguinte reproduzirá esta mesma seleção por indexação explícita.
print(f"\nmédia: {average.item():+.3f}")

prompt 0: +3.337 → favorece o nome correto
prompt 1: +3.202 → favorece o nome correto
prompt 2: +2.709 → favorece o nome correto
prompt 3: +3.797 → favorece o nome correto
prompt 4: +1.720 → favorece o nome correto
prompt 5: +5.281 → favorece o nome correto
prompt 6: +2.601 → favorece o nome correto
prompt 7: +5.767 → favorece o nome correto

média: +3.552


In [6]:
# Confirmamos os eixos usados acima: logits compartilham (batch, position) com `tokens`.
assert logits.shape[:2] == tokens.shape
assert answer_tokens.shape == (8, 2)

# `prompt_indices` pareia cada linha do batch com seus próprios IDs de resposta, sem cruzar exemplos.
prompt_indices = torch.arange(len(prompts), device=device)
# A indexação avançada reúne um logit por prompt na posição -1: primeiro correto, depois incorreto.
manual = (
    logits[prompt_indices, -1, answer_tokens[:, 0]]
    - logits[prompt_indices, -1, answer_tokens[:, 1]]
)

# Esta checagem fecha o baseline que o Notebook 02 receberá para decomposição.
torch.testing.assert_close(per_prompt, manual)
assert average.item() > 0
print("Verificações concluídas: a implementação e o cálculo manual concordam.")

Verificações concluídas: a implementação e o cálculo manual concordam.


## Interpretação

Uma diferença média positiva mostra que, neste conjunto, o modelo tende a favorecer o nome correto. Os valores individuais evitam confundir essa média com acerto garantido em todo prompt.

Isso mede o **comportamento final**, mas ainda não revela quais heads participam do resultado. O Notebook 02 recebe daqui três peças concretas: o baseline médio, a diferença por prompt e a convenção `answer_tokens[:, 0]` para correto versus `answer_tokens[:, 1]` para incorreto. Com elas, poderá decompor o mesmo resultado sem mudar a pergunta medida.